# Concierge Planning Agent: Multi-Agent Personal Scheduler & Progress Tracker
**Kaggle AI Agents Intensive Capstone Project** | **Track:** Concierge Agents  
*Reference:* [Agents Intensive Capstone Project Overview](https://www.kaggle.com/competitions/agents-intensive-capstone-project/overview)

---

### Executive Pitch & Overview

#### The Problem
Self-directed learning and personal project execution suffer from high failure rates. Learners frequently encounter:
1. **Unrealistic Planning**: Overestimating daily capacity leads to immediate schedule breakdown.
2. **Scope Creep**: Lack of structured milestones causes goal drift.
3. **Burnout & Abandonment**: Static calendar tools do not adapt when a user falls behind, leading to frustration and abandoned projects.

#### The Solution
The **Concierge Planning Agent** is an autonomous multi-agent system built using Gemini 2.5 Flash and Pydantic. It acts as an intelligent personal project manager that:
- Captures and validates user constraints (interests, skill level, daily time budget).
- Recommends tailored, step-by-step project blueprints calibrated to available daily hours.
- Enforces safety controls to prevent burnout (max 1 active project at a time).
- Dynamically reconfigures deadlines and reschedules remaining tasks when the user falls behind.
- Maintains long-term memory of completed projects and user growth over time.

## 1. Setup & Environment Dependencies

First, we install the required packages: `google-genai`, `pydantic`, `python-dotenv`, and `streamlit`.

In [ ]:
!pip install -q google-genai pydantic python-dotenv streamlit

## 2. API Key & Environment Configuration
Load `GEMINI_API_KEY` securely from the environment or `.env` file without leaking sensitive keys into the code.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Check for API key
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    print("WARNING: GEMINI_API_KEY is not set. Please set os.environ['GEMINI_API_KEY'] = 'your_key' before running LLM queries.")
else:
    print("SUCCESS: GEMINI_API_KEY detected.")

## 3. Data Schemas (Pydantic Models)
We use Pydantic models to enforce strict, type-safe JSON output structures from Gemini models.

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional

class UserPreferences(BaseModel):
    interests: List[str] = Field(..., description="List of topics or technologies the user is interested in.")
    hours_per_day: float = Field(..., description="Number of hours available per day (0.5 to 16).")
    skill_level: str = Field(..., description="Skill level: Beginner, Intermediate, or Advanced.")
    target_duration_days: int = Field(..., description="Project duration in days.")

class DailyTask(BaseModel):
    day: int = Field(..., description="Day number of the project (1-indexed).")
    title: str = Field(..., description="Task title.")
    description: str = Field(..., description="Task description.")
    estimated_hours: float = Field(..., description="Estimated hours to complete.")
    completed: bool = Field(default=False, description="Completion status.")

class ProjectProposal(BaseModel):
    name: str = Field(..., description="Creative project name.")
    description: str = Field(..., description="High-level project summary.")
    difficulty: str = Field(..., description="Assessed difficulty level.")
    total_days: int = Field(..., description="Duration in days.")
    daily_tasks: List[DailyTask] = Field(..., description="Milestone tasks for each day.")

class ReconfiguredSchedule(BaseModel):
    justification: str = Field(..., description="Explanation of schedule adjustments.")
    revised_tasks: List[DailyTask] = Field(..., description="Revised remaining tasks.")

## 4. Multi-Agent Architecture Implementation

We implement 5 distinct agents:
1. `BaseAgent`: Base class for Gemini client initialization and structured LLM generation.
2. `PreferenceAgent`: Input parsing and constraint validation.
3. `ProjectRecommendationAgent`: Generates custom blueprints using Gemini structured outputs.
4. `MemoryAgent`: Handles local state (`memory_store.json`), active project tracking, and profile summaries.
5. `ProgressAgent`: Analyzes completion rate and reschedules remaining tasks.
6. `PlannerAgent`: Root orchestrator managing state transitions and burnout safety rules.

In [ ]:
import json
from google import genai
from google.genai import types

class BaseAgent:
    def __init__(self, name: str, system_instruction: str):
        self.name = name
        self.system_instruction = system_instruction
        self._client = None
        self._model = "gemini-2.5-flash"

    def get_client(self):
        if self._client is not None:
            return self._client
        api_key = os.getenv("GEMINI_API_KEY")
        if not api_key:
            raise ValueError(f"[{self.name}] GEMINI_API_KEY is not configured.")
        self._client = genai.Client(api_key=api_key)
        return self._client

    def log(self, message: str):
        print(f"[{self.name}] {message}")

    def generate_structured(self, prompt: str, schema):
        self.log(f"Generating structured response matching schema '{schema.__name__}'...")
        client = self.get_client()
        response = client.models.generate_content(
            model=self._model,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=self.system_instruction,
                response_mime_type="application/json",
                response_schema=schema,
                temperature=0.2,
            )
        )
        return response.parsed

    def generate_chat(self, prompt: str) -> str:
        client = self.get_client()
        response = client.models.generate_content(
            model=self._model,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=self.system_instruction,
                temperature=0.7,
            )
        )
        return response.text

# 1. Memory Agent
class MemoryAgent(BaseAgent):
    def __init__(self, db_path="memory_store.json"):
        super().__init__("Memory Agent", "You manage persistent memory, project history, and user growth summaries.")
        self.db_path = db_path
        self._init_db()

    def _init_db(self):
        if not os.path.exists(self.db_path):
            self.save_state({
                "preferences": None,
                "active_project": None,
                "project_history": [],
                "user_summary": "New user with no project history."
            })

    def load_state(self):
        with open(self.db_path, "r", encoding="utf-8") as f:
            return json.load(f)

    def save_state(self, state):
        with open(self.db_path, "w", encoding="utf-8") as f:
            json.dump(state, f, indent=4)

    def get_preferences(self):
        state = self.load_state()
        p = state.get("preferences")
        return UserPreferences(**p) if p else None

    def save_preferences(self, prefs: UserPreferences):
        state = self.load_state()
        state["preferences"] = prefs.model_dump()
        self.save_state(state)

    def get_active_project(self):
        state = self.load_state()
        p = state.get("active_project")
        return ProjectProposal(**p) if p else None

    def save_active_project(self, project: ProjectProposal):
        state = self.load_state()
        state["active_project"] = project.model_dump()
        self.save_state(state)

    def abandon_active_project(self, reason="Abandoned by user"):
        state = self.load_state()
        p = state.get("active_project")
        if p:
            p["status"] = "Abandoned"
            p["abandon_reason"] = reason
            state["project_history"].append(p)
            state["active_project"] = None
            self.save_state(state)

    def complete_active_project(self):
        state = self.load_state()
        p = state.get("active_project")
        if p:
            p["status"] = "Completed"
            state["project_history"].append(p)
            state["active_project"] = None
            self.save_state(state)

# 2. Preference Agent
class PreferenceAgent(BaseAgent):
    def __init__(self):
        super().__init__("Preference Agent", "Extract and validate user learning preferences and constraints.")

    def parse_preferences(self, user_input: str) -> UserPreferences:
        prompt = f"Extract preferences into structured format from user input: '{user_input}'."
        return self.generate_structured(prompt, UserPreferences)

    def validate_preferences(self, prefs: UserPreferences) -> tuple[bool, str]:
        if not prefs.interests:
            return False, "At least one interest is required."
        if prefs.hours_per_day < 0.5 or prefs.hours_per_day > 16.0:
            return False, "Hours per day must be between 0.5 and 16."
        if prefs.target_duration_days <= 0:
            return False, "Duration must be at least 1 day."
        return True, ""

# 3. Recommendation Agent
class ProjectRecommendationAgent(BaseAgent):
    def __init__(self):
        super().__init__("Recommendation Agent", "Design tailored project blueprints and daily tasks.")

    def recommend_project(self, prefs: UserPreferences, declined_projects: list = None) -> ProjectProposal:
        declined = ", ".join(declined_projects) if declined_projects else "None"
        prompt = (
            f"Generate a project proposal for:\n"
            f"- Interests: {', '.join(prefs.interests)}\n"
            f"- Skill: {prefs.skill_level}\n"
            f"- Time: {prefs.hours_per_day} hours/day\n"
            f"- Duration: {prefs.target_duration_days} days\n"
            f"- Avoid declined projects: {declined}\n"
            f"Generate exactly {prefs.target_duration_days} days of tasks."
        )
        return self.generate_structured(prompt, ProjectProposal)

# 4. Progress Agent
class ProgressAgent(BaseAgent):
    def __init__(self):
        super().__init__("Progress Agent", "Analyze progress and reconfigure remaining deadlines dynamically.")

    def analyze_and_reconfigure(self, prefs: UserPreferences, project: ProjectProposal, current_day: int) -> ReconfiguredSchedule:
        tasks_summary = "\n".join([f"- Day {t.day}: {t.title} | Completed: {t.completed}" for t in project.daily_tasks])
        prompt = (
            f"Project: {project.name}\n"
            f"Daily hours: {prefs.hours_per_day}\n"
            f"Current Day: {current_day} of {project.total_days}\n"
            f"Task status:\n{tasks_summary}\n\n"
            f"Reconfigure remaining pending tasks from Day {current_day} to Day {project.total_days} without changing project duration."
        )
        return self.generate_structured(prompt, ReconfiguredSchedule)

# 5. Root Planner Orchestrator
class PlannerAgent(BaseAgent):
    def __init__(self):
        super().__init__("Planner Orchestrator", "Root orchestrator coordinating all sub-agents and enforcing safety rules.")
        self.memory_agent = MemoryAgent()
        self.preference_agent = PreferenceAgent()
        self.recommendation_agent = ProjectRecommendationAgent()
        self.progress_agent = ProgressAgent()

    def process_preferences(self, raw_input: str):
        if self.memory_agent.get_active_project():
            return False, "You already have an active project. Complete or abandon it first."
        prefs = self.preference_agent.parse_preferences(raw_input)
        is_valid, err = self.preference_agent.validate_preferences(prefs)
        if not is_valid:
            return False, err
        self.memory_agent.save_preferences(prefs)
        return True, "Preferences updated."

    def get_recommendation(self, declined=None):
        if self.memory_agent.get_active_project():
            return None, "Active project exists! Single project limit enforced to prevent burnout."
        prefs = self.memory_agent.get_preferences()
        if not prefs:
            return None, "Please set preferences first."
        return self.recommendation_agent.recommend_project(prefs, declined), ""

    def accept_project(self, project: ProjectProposal):
        if self.memory_agent.get_active_project():
            return False, "Active project exists!"
        self.memory_agent.save_active_project(project)
        return True, f"Project '{project.name}' activated!"

## 5. End-to-End System Simulation & Demonstration

Let's run a complete simulation of the multi-agent system handling user preferences, generating recommendations, activating a project, simulating falling behind, and dynamically reconfiguring deadlines.

In [ ]:
# Initialize root orchestrator
planner = PlannerAgent()

# Step 1: Input user preferences
print("--- Step 1: Processing Preferences ---")
raw_input = "I want to learn Python and Web Scraping. I have 2 hours per day for a 3-day project. Skill level beginner."
success, msg = planner.process_preferences(raw_input)
print(f"Status: {success} | Message: {msg}")

# Step 2: Request project recommendation
print("\n--- Step 2: Requesting Project Proposal ---")
proposal, error = planner.get_recommendation()
if proposal:
    print(f"Project Name: {proposal.name}")
    print(f"Description: {proposal.description}")
    print(f"Total Days: {proposal.total_days}")
    for task in proposal.daily_tasks:
        print(f"  Day {task.day}: {task.title} ({task.estimated_hours}h) - {task.description}")

# Step 3: Accept project proposal
print("\n--- Step 3: Accepting Project ---")
if proposal:
    acc_success, acc_msg = planner.accept_project(proposal)
    print(f"Accept Status: {acc_success} | Message: {acc_msg}")

# Step 4: Test Single Active Project Burnout Prevention Rule
print("\n--- Step 4: Testing Safety Rule (Attempting second project) ---")
second_prop, second_err = planner.get_recommendation()
print(f"Second Recommendation Result: {second_err}")

# Step 5: Simulate Progress & Reconfiguration
print("\n--- Step 5: Simulating Schedule Reconfiguration ---")
active = planner.memory_agent.get_active_project()
if active:
    # Simulate Day 1 complete, Day 2 pending, currently on Day 3
    active.daily_tasks[0].completed = True
    active.daily_tasks[1].completed = False
    active.daily_tasks[2].completed = False
    planner.memory_agent.save_active_project(active)
    
    prefs = planner.memory_agent.get_preferences()
    reconfigured = planner.progress_agent.analyze_and_reconfigure(prefs, active, current_day=3)
    print(f"Reconfiguration Justification: {reconfigured.justification}")
    print("Revised Remaining Tasks:")
    for task in reconfigured.revised_tasks:
        print(f"  Day {task.day}: {task.title} ({task.estimated_hours}h) - {task.description}")

## 6. Rubric & Evaluation Mapping

| Kaggle Criteria | Implementation Summary |
| :--- | :--- |
| **Pitch & Value** | Solves personal project burnout and scope creep with tailored daily task sizing and adaptive schedule re-budgeting. |
| **Multi-Agent Architecture** | 5 coordinated agents (`Planner`, `Preference`, `Recommendation`, `Memory`, `Progress`). |
| **Structured Outputs** | Pydantic model schemas enforced via Gemini API `response_schema`. |
| **Persistent Memory** | `MemoryAgent` persists project state, task updates, and history in `memory_store.json`. |
| **Safety Controls** | Single-project limit enforced to prevent burnout; clean API key environment handling. |